# Validation — does this system actually have an edge?

Gate 1 found the videos. Gate 2 pulled the transcripts. The rules were extracted
and verified against them. This notebook answers the only question that matters
next: **when those rules are traded mechanically over history you did not tune
them on, do they make money?**

It is designed to be able to say no. Most of what it prints is there to catch a
result that looks good for the wrong reason.

---

## What you need

A **1-minute CSV** for one instrument. The more history the better:

| What you want to learn | Minimum |
| --- | --- |
| That the pipeline runs | ~2 weeks |
| Whether a rule is any good | **2 years** |

Two weeks will run and print numbers. Those numbers will mean nothing — the
trade windows are about an hour a day, and a setup does not appear every day.
Small samples here do not give a cautious answer, they give a confident wrong
one.

Where to get the data: `data/bars/README.md` in the repo lists sources. Your own
MT4/MT5 broker is free and is the closest match to what you would really have
traded — but note it exports in **broker server time**, not UTC, and Step 4
below exists to catch exactly that.

---

## How to run it

**Runtime → Run all**, then follow the upload prompt in Step 3.

## Step 1 — Install

In [ ]:
%pip install -q --upgrade pydantic
print("Step 1 done.")

## Step 2 — Get the code

In [ ]:
import os, shutil, subprocess

REPO = "https://github.com/dboy140/Dboytrades.git"
BRANCH = "claude/ict-nbbtrader-trading-system-43hipg"

if os.path.isdir("/content/Dboytrades"):
    shutil.rmtree("/content/Dboytrades")

r = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/Dboytrades"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    raise SystemExit(f"Could not download the code:\n{r.stderr}")

os.chdir("/content/Dboytrades")

t = subprocess.run(["python", "-m", "pytest", "-q"], capture_output=True, text=True)
print(t.stdout.strip().splitlines()[-1] if t.stdout else t.stderr[-300:])
print("\nStep 2 done.")

## Step 3 — Upload your CSV

Click **Choose Files** when the button appears. One file, 1-minute bars.

Expected columns: a timestamp, then open, high, low, close, volume. The
timestamp must carry a timezone (`Z` or an offset) — the loader refuses naive
timestamps rather than assuming UTC, because a guessed timezone silently
invalidates every session rule in the system.

In [ ]:
import pathlib, shutil

CSV = None
try:
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    pathlib.Path("data/bars").mkdir(parents=True, exist_ok=True)
    CSV = f"data/bars/{name}"
    shutil.move(name, CSV)
except Exception as exc:
    print(f"Upload widget unavailable ({exc}).")
    print("Use the folder icon in the left sidebar to upload into data/bars/,")
    print("then set CSV below by hand.")

print("CSV =", CSV)

## Step 4 — Is the data what it claims to be?

**Do not skip this.** MT4/MT5 exports are stamped in broker server time, usually
UTC+2 or UTC+3. Label those as UTC and every session window is wrong by two or
three hours — and the backtest will run anyway and print confident numbers.

This check does not trust the label. It finds the daily volatility cycle and
derives the real offset. On short files it reports INCONCLUSIVE rather than
guessing.

In [ ]:
!python -m bot.inspect_data "$CSV" 

### If it reported a drift

Uncomment the line below, set the number it reported, and run it. This writes a
`.fixed.csv` and leaves your original alone. Then re-run Step 4 on the fixed
file to confirm the drift is now 0.

Leave it commented if Step 4 reported no drift.

In [ ]:
# !python -m bot.inspect_data "$CSV" --restamp -3
# CSV = CSV.replace(".csv", ".fixed.csv"); print("now using", CSV)

## Step 5 — Pick the instrument and setup

- **`ote`** — LDN to New York OTE. Derives its own direction from the market
  structure shift, so it needs no bias from you.
- **`silver_bullet`** — needs a higher timeframe bias. That bias is *not*
  mechanically derivable from this corpus (GAPS G-07), so the engine refuses to
  guess it. Set `BIAS` to `long` or `short` and understand you are testing one
  half of the picture.

In [ ]:
INSTRUMENT = "EURUSD"      # NAS100 | EURUSD | GBPUSD | XAUUSD
SETUP      = "ote"         # ote | silver_bullet
BIAS       = "long"        # only used by silver_bullet

print(f"{SETUP} on {INSTRUMENT}")

## Step 6 — Backtest

Every fill here is pessimistic on purpose. Entries are limit orders, not market
orders, because that is what the entry rules describe. When a bar contains both
the stop and the target, the stop is assumed. Excursion is capped at the exit,
so a 1R loss reports as −1R rather than however far price ran afterwards.

In [ ]:
args = ["python", "-m", "bot.run_backtest", CSV,
        "--instrument", INSTRUMENT, "--setup", SETUP]
if SETUP == "silver_bullet":
    args += ["--bias", BIAS]

import subprocess
p = subprocess.run(args, capture_output=True, text=True)
print(p.stdout or p.stderr)

## Step 7 — Walk-forward validation

The real test. The history is cut into consecutive blocks; parameters are chosen
on each training block and then traded, untouched, on the block that follows.

A backtest tuned and scored on the same data will look good no matter what — it
is measuring how well the tuning fitted the past, not whether anything works.
This measures how much of that survives contact with data the tuning never saw.

This is the slower cell. On two years of 1-minute data expect a few minutes.

In [ ]:
args = ["python", "-m", "bot.run_validate", CSV,
        "--instrument", INSTRUMENT, "--setup", SETUP,
        "--bias", BIAS, "--out", "logs/validation.json"]

import subprocess
p = subprocess.run(args, capture_output=True, text=True)
print(p.stdout or p.stderr)

## How to read what Step 7 printed

**The verdict**

| Verdict | Meaning |
| --- | --- |
| `FAILED` | Out-of-sample expectancy is negative. It does not work. This is the most likely outcome and it is a real result, not a bug. |
| `NOT DISTINGUISHABLE FROM LUCK` | Positive on average, but the 95% interval includes zero. A system with no edge produces results like this routinely. |
| `INCONSISTENT` | Profitable in only a minority of windows — more likely a property of those periods than of the rules. |
| `WEAK` | Positive, interval clears zero, but most of the in-sample edge was lost. |
| `INSUFFICIENT DATA` | Too few out-of-sample trades to say anything. Get more history. Not a failure — a shortage. |
| `SURVIVED out of sample` | It held up on all counts. **Necessary, not sufficient** — see below. |

**Why this is hard to pass on purpose.** An earlier version of this check asked
only for 20 out-of-sample trades and a positive average. A pure random walk —
data with no edge in it whatsoever — cleared it and was declared `SURVIVED`,
which was enough to unlock live trading. That is the worst way for a validator
to be wrong, because it fails towards risking money. It now also requires the
confidence interval to exclude zero and most windows to be profitable.

**"Is the out-of-sample result distinguishable from luck?"** — a bootstrap
interval on the pooled out-of-sample trades. If `positive_with_95pct_confidence`
is `False`, a positive average is consistent with having been lucky.

**"What drawdown should be expected?"** — the same trades reshuffled thousands of
times. The drawdown that happened was one ordering out of many; this shows the
range to be prepared for. It is usually worse than the one in the report.

**"Parameter sensitivity"** — every setting tried, not just the best. If only one
or two make money and the rest lose, the winner was almost certainly selected to
fit this history. That is `curve-fitting suspected: True`, and a high win rate
alongside it is a warning, not a result.

---

### If it survived

That is the beginning of the evidence, not the end of it. Walk-forward removes
some specific ways of fooling yourself. It does not account for spread,
commission, slippage, or the fact that a strategy chosen after looking at
several is already selected on this data at one remove.

Live mode is gated on this report for that reason — `bot/live.py` refuses to
place a real order unless out-of-sample expectancy is positive over 30+ trades,
the interval excludes zero, and most windows were profitable. That gate is a
floor, not a green light.

### If it failed

That is worth knowing and it cost you nothing. Do not tune settings until it
passes: with enough attempts something always passes, and what you will have
built is a record of this file's noise. The honest responses are more data, a
different instrument, or accepting that this particular rule does not carry.

## Step 8 — Download the report

In [ ]:
import pathlib
p = pathlib.Path("logs/validation.json")
if not p.exists():
    print("No report -- Step 7 did not finish. Check its output above.")
else:
    print(p.read_text()[:2000])
    try:
        from google.colab import files
        files.download(str(p))
    except Exception as exc:
        print(f"\nAuto-download unavailable ({exc}).")
        print("Grab logs/validation.json from the folder icon in the sidebar.")

---

## Paste back into the chat

1. The **VERDICT** line and the fold table from Step 7
2. The **parameter sensitivity** block, especially `curve-fitting suspected`
3. Anything Step 4 said about your timestamps

A `FAILED` verdict is as useful to report as a passing one — more so, because it
is the one that stops money being lost.